In [ ]:
import tensorflow as tf
import numpy as np
import tifffile as tiff
import os

# ===== 配置 =====
batch_size = 32
isprs_dir = r'D:/yrq/Semantic_segmentation/Gid'
image_size = (256, 256)

# ===== 颜色映射 =====
COLORMAP_16c = [
    [0, 0, 0], [200, 0, 0], [250, 0, 150], [200, 150, 150], [250, 150, 150],
    [0, 200, 0], [150, 250, 0], [150, 200, 150], [200, 0, 200],
    [150, 0, 250], [150, 150, 250], [250, 200, 0], [200, 200, 0], [0, 0, 200],
    [0, 150, 200], [0, 200, 250]
]
COLORMAP_6c = [
    [0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 255, 255], [255, 255, 0], [0, 0, 255]
]

# ===== 构建映射表 =====
def build_colormap_table(colormap):
    table = np.zeros(256 ** 3)
    for i, color in enumerate(colormap):
        table[(color[0] * 256 + color[1]) * 256 + color[2]] = i
    return tf.convert_to_tensor(table)

colormap2label_16 = build_colormap_table(COLORMAP_16c)
colormap2label_6 = build_colormap_table(COLORMAP_6c)

# ===== 标签转换函数 =====
def label_indices(label_img, colormap2label):
    label_img = tf.cast(label_img, tf.int32)
    idx = ((label_img[:, :, 0] * 256 + label_img[:, :, 1]) * 256 + label_img[:, :, 2])
    return tf.gather(colormap2label, idx)

# ===== 读取文件列表 =====
def get_file_paths(is_train=True):
    txt_path = os.path.join(isprs_dir, 'tifs', 'train_data.txt' if is_train else 'val_data.txt')
    with open(txt_path, 'r') as f:
        names = f.read().split()
    imgs = [os.path.join(isprs_dir, 'srcs', 'srcs_train', n) for n in names]
    labels16 = [os.path.join(isprs_dir, 'label_15cs', 'labels_train', n) for n in names]
    labels6 = [os.path.join(isprs_dir, 'label_5cs', 'labels_train', n) for n in names]
    return imgs, labels16, labels6

# ===== 数据生成器 =====
def data_generator(img_paths, lbl16_paths, lbl6_paths):
    for img_p, lbl16_p, lbl6_p in zip(img_paths, lbl16_paths, lbl6_paths):
        img = tiff.imread(img_p).astype(np.float32)
        lbl16 = tiff.imread(lbl16_p)
        lbl6 = tiff.imread(lbl6_p)

        # 转Tensor
        img_tf = tf.convert_to_tensor(img)
        lbl16_tf = tf.convert_to_tensor(lbl16)
        lbl6_tf = tf.convert_to_tensor(lbl6)

        # 标签索引化
        lbl16_idx = label_indices(lbl16_tf, colormap2label_16)
        lbl6_idx = label_indices(lbl6_tf, colormap2label_6)

        yield img_tf, (lbl16_idx, lbl6_idx)

# ===== 创建 Dataset =====
def create_dataset(is_train=True, batch_size=8):
    imgs, lbl16, lbl6 = get_file_paths(is_train)
    print(f"{'训练集' if is_train else '测试集'} 样本数量: {len(imgs)}")  # ✅ 打印数量
    dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(imgs, lbl16, lbl6),
        output_signature=(
            tf.TensorSpec(shape=(256, 256, 3), dtype=tf.float32),
            (
                tf.TensorSpec(shape=(256, 256), dtype=tf.float32),
                tf.TensorSpec(shape=(256, 256), dtype=tf.float32)
            )
        )
    )
    if is_train:
        dataset = dataset.shuffle(buffer_size=1024)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# ===== 获取训练集 & 测试集 =====
train_dataset = create_dataset(is_train=True, batch_size=batch_size)
test_dataset = create_dataset(is_train=False, batch_size=batch_size)

print("✅ 数据集创建完成，可以开始训练")
